In [3]:
# Instalando o experta e ajustando a dependência do frozendict
!pip install experta
!pip install frozendict==1.2

In [4]:
# Imports
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping

from experta import KnowledgeEngine, Rule, Fact, MATCH, AS, P, AND, OR, NOT

In [5]:
# --- MODELAGEM DO DOMÍNIO: LIBRAS ---

class ParametroSinal(Fact):
    """
    Recebe os dados físicos do sinal bruto.
    Campos esperados: mao (configuração), local (ponto de articulação), movimento.
    """
    pass

class Expressao(Fact):
    """
    Recebe a expressão facial ou corporal.
    Campos esperados: tipo (exemplos: neutra, interrogativa, dor).
    """
    pass

class Contexto(Fact):
    """
    Fato intermediário (Nível 2). Gerado após interpretar os parâmetros físicos.
    Campos esperados: categoria (exemplos: tempo, sentimento, cognitivo).
    """
    pass

class SignificadoFinal(Fact):
    """
    A saída final do sistema (Nível 3).
    Campos esperados: traducao (a palavra ou frase em português).
    """
    pass

In [6]:
# --- TESTANDO A MODELAGEM ---
sinal_teste = ParametroSinal(mao="em_C", local="testa", movimento="nenhum")
print("Validando a criação do Fato:")
print(f"Acessando o local de articulação: {sinal_teste['local']}")

Validando a criação do Fato:
Acessando o local de articulação: testa


In [7]:
class ClassificadorLibras(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        # Lista para guardar o "trace" (o rastro de decisões)
        self.trace_decisoes = []

    # REGRAS DE NÍVEL 1: Deduzindo o contexto (parâmetro -> contexto)

    @Rule(ParametroSinal(mao="em_C", local="testa"))
    def contexto_cognitivo(self):
        motivo = "Regra Nível 1: Mão em 'C' na testa -> Contexto 'Cognitivo/Mental'"
        self.trace_decisoes.append(motivo)
        print(f"[DISPARO] {motivo}")
        # Regra dispara e joga novo fato na memória de trabalho
        self.declare(Contexto(categoria="cognitivo"))

    @Rule(ParametroSinal(mao="aberta", local="peito"))
    def contexto_sentimento(self):
        motivo = "Regra Nível 1: Mão aberta no peito -> Contexto 'Sentimento/Emoção'"
        self.trace_decisoes.append(motivo)
        print(f"[DISPARO] {motivo}")
        self.declare(Contexto(categoria="sentimento"))

    @Rule(ParametroSinal(mao="em_L", local="espaco_neutro"))
    def contexto_tempo(self):
        motivo = "Regra Nível 1: Mão em 'L' no espaço neutro -> Contexto 'Tempo/Calendário'"
        self.trace_decisoes.append(motivo)
        print(f"[DISPARO] {motivo}")
        self.declare(Contexto(categoria="tempo"))

In [8]:
# --- Testando o nível 1 ---
engine = ClassificadorLibras()
engine.reset()  # Limpa a memória de trabalho antes de começar

# Fato inicial injetado: o que a câmera ou luva leu do usuário
engine.declare(ParametroSinal(mao="em_C", local="testa", movimento="nenhum"))

# Inicia o raciocínio
print("Iniciando o raciocínio do motor...\n")
engine.run()

print("\n--- RESUMO ---")
print("Rastro de decisões (Trace):", engine.trace_decisoes)
print("\nFatos que estão na memória final:")
for f_id, f_data in engine.facts.items():
    print(f"ID {f_id}: {f_data}")

Iniciando o raciocínio do motor...

[DISPARO] Regra Nível 1: Mão em 'C' na testa -> Contexto 'Cognitivo/Mental'

--- RESUMO ---
Rastro de decisões (Trace): ["Regra Nível 1: Mão em 'C' na testa -> Contexto 'Cognitivo/Mental'"]

Fatos que estão na memória final:
ID 0: <f-0>
ID 1: <f-1>
ID 2: <f-2>
